# Semi-Supervised ML Tagging — Example Notebook

This notebook demonstrates **semi-supervised learning** for AS tagging when you have only a small labeled set (tens of positive and optionally negative ASes).

## When to Use Semi-Supervised

- You can only manually verify and label on the order of tens of ASes.
- You want to either assign tags directly or bootstrap feature exploration for future supervised learning.

## PUN Model

Our framework supports **PUN** (Positive, Unlabeled, small Negative), inspired by PUbN (Hsieh et al.). PUN combines:
- **Graph score**: PPR-style propagation from positive seeds on AS topology.
- **Feature score**: LogReg (with negatives), One-Class SVM, or Autoencoder (positive-only).
- **Combined**: Weighted sum of graph + feature scores.

---

## Installation

Same as supervised ML tagging: `pip install as-tagging[ml]`

In [ ]:
# --- DGL troubleshooting: run this if you see graphbolt or DILL_AVAILABLE errors ---
import sys
print("Python:", sys.executable)
import torch
print("PyTorch:", torch.__version__)
# Patch for PyTorch 2.3.0 + torchdata: DILL_AVAILABLE was removed from torch
_common = torch.utils.data.datapipes.utils.common
if not hasattr(_common, "DILL_AVAILABLE"):
    try:
        _common.DILL_AVAILABLE = torch.utils._import_utils.dill_available()
    except (AttributeError, ImportError):
        try:
            import dill
            _common.DILL_AVAILABLE = True
        except ImportError:
            _common.DILL_AVAILABLE = False
try:
    import dgl
    print("DGL:", dgl.__version__)
    import dgl.graphbolt  # This triggers the graphbolt load
    print("GraphBolt: OK")
except Exception as e:
    print("Error:", e)
    print("\nFix: In a terminal, activate the SAME env as this notebook, then:")
    print("  pip install pydantic \"typing-extensions>=4.14.0\"")
    print("  # If graphbolt errors: pip uninstall torch torchdata dgl -y")
    print("  #   pip install torch==2.3.0 torchdata==0.7.1")
    print("  #   pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/repo.html")
    print("Restart the kernel and re-run.")

---
## Step 1: Load Snapshot Data

Choose Online or Offline mode.

In [ ]:
# Option A: Offline (set DATA_PATH to your snapshot directory)
from as_tagging import ASTagging, OfflineSnapshotProvider

DATA_PATH = "/path/to/as_feature_snapshots"  # Update to your path
DATE = "2026-01"

offline_provider = OfflineSnapshotProvider(DATA_PATH)
tagger = ASTagging(snapshot_provider=offline_provider, date=DATE)
print(f"Loaded {len(tagger.atomic_tags)} ASNs")

# Option B: Online (requires HF_TOKEN for HuggingFace)
# from as_tagging import ASTagging, OnlineSnapshotProvider
# import os
# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# provider = OnlineSnapshotProvider(token=os.environ.get("HF_TOKEN"))
# tagger = ASTagging(snapshot_provider=provider, date=DATE)

---
## Step 2: Prepare Labels (Small Set)

**positive_asns** (required): ASNs with the target property.
**negative_asns** (optional): ASNs without it. Can be few or empty; OCC/AE work with positives only.

Example: Mobile ISPs vs non-mobile eyeball ASes.

In [ ]:
# Small labeled set for Mobile ISP tagging in ARIN and LACNIC (10 positive, 29 negative)
positive_asns = [10396, 13771, 33392, 15344, 36511, 35900, 6639, 30689, 396357, 11594]
positive_asns += [14988, 15022, 15964, 21271, 24835] # AFRINIC (15022 from multi-as org Adept Internet)
positive_asns += [2518, 3605, 4817, 17893, 131284] # APNIC (2518 from multi-as org BIGLOBE)
positive_asns += [2606, 3238, 6752, 12620, 13122] # RIPE

negative_asns = list(set([19016, 265524, 16832, 30640, 134053, 15054, 
39611, 46920, 38566, 262591, 31726, 3356, 201004, 17501, 134371, 36924, 3356, 30058, 
6677, 61461, 203214, 197558, 18049]))
# [8070, 36231, 54665, 5650, 27924, 13335, 16413, 19237, 268581, 52263, 27947, 27694, 16437, 393275, 394684, 36290, 15305, 209, 22995, 263763, 30165, 22363, 3549, 19551, 21859, 14061, 52468, 19323, 16509]


print(f"{len(positive_asns)} positive, {len(negative_asns)} negative")

---
## Step 3: Assign Tag via Semi-Supervised (One-Line API)

In [ ]:
tagger.AssignTag(
    tag_name="ARIN or LACNIC ASes",
    expression=lambda tags: tags.get('delegation_rir', "") == "arin" or tags.get('delegation_rir', "") == "lacnic"
)
arin_lacnic_ases = tagger.FetchTag("ARIN or LACNIC ASes")

eyeball_asns = tagger.ListASNsWithoutTag("No Eyeball", treat_false_as_missing=True)
arin_lacnic_eyeball_asns = list(set(arin_lacnic_ases).intersection(eyeball_asns))
print("arin_lacnic_ases", len(arin_lacnic_ases))
print("arin_lacnic_eyeball_asns", len(arin_lacnic_eyeball_asns))

In [ ]:
results = tagger.AssignSemiSupervisedMLTag(
    tag_name="Mobile ISP",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    model="pun",
    pun_method="combined",  # graph_ppr, logreg, occ, ae, or combined
    threshold=0.5,
    # target_precision=0.9,  # Optional: tune threshold for high precision
    verbose=True,
    # asns=arin_lacnic_eyeball_asns,
)

print(f"Model: {results['model']}, Tagged: {results['n_tagged']}")

---
## Advanced: Low-Level API

In [ ]:
from as_tagging.ml import SemiSupervisedMLTagger

# Pass manifest/schema when available so feature detection matches AssignSemiSupervisedMLTag (71 vs 70)
manifest = None
snapshot_schema = None
if hasattr(tagger, "snapshot_provider") and tagger.snapshot_provider is not None:
    if hasattr(tagger.snapshot_provider, "get_manifest"):
        try:
            manifest = tagger.snapshot_provider.get_manifest(tagger.date)
        except Exception:
            pass
    if hasattr(tagger.snapshot_provider, "get_schema"):
        try:
            snapshot_schema = tagger.snapshot_provider.get_schema(tagger.date)
        except Exception:
            pass

ss = SemiSupervisedMLTagger(
    snapshot_dict=tagger.atomic_tags,
    model="pun",
    pun_method="combined",  # or graph_ppr, logreg, occ, ae
    manifest=manifest,
    snapshot_schema=snapshot_schema,
)

# Advanced PU+N knobs (new defaults shown explicitly)
# - unlabeled_weight: weight of unlabeled examples as weak negatives in the feature model
# - graph_neg_weight: subtractive weight for negative-seed graph diffusion
# - tune_blend_weight: auto-tune w_feat/w_graph on labeled set
# - blend_grid: candidate feature weights (graph weight = 1 - w_feat)
# - use_calibration: sigmoid calibration for LR probabilities when possible
ss.fit(
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    unlabeled_weight=0.1,
    graph_neg_weight=2.0,
    tune_blend_weight=True,
    blend_grid=[0.0, 0.25, 0.5, 0.75, 1.0],
    use_calibration=True,
)

In [ ]:
ss = SemiSupervisedMLTagger(
    snapshot_dict=tagger.atomic_tags,
    model="pun",
    pun_method="logreg",   # <- turns graph off
    manifest=manifest,
    snapshot_schema=snapshot_schema,
)
ss.fit(positive_asns=positive_asns, negative_asns=negative_asns)

In [ ]:
unlabeled_arin_lacnic_ases = set(arin_lacnic_eyeball_asns).difference(positive_asns).difference(negative_asns)
print(f"Unlabeled ASes: {len(arin_lacnic_eyeball_asns)}")

In [ ]:
unlabeled_ases = set(eyeball_asns).difference(positive_asns).difference(negative_asns)
print(f"Unlabeled ASes: {len(unlabeled_ases)}")

In [ ]:
scores = ss.predict(asns=list(unlabeled_ases))
top_100 = dict(sorted(scores.items(), key=lambda x: -x[1])[:100])
tags = {a: True for a in top_100}  # only assign True to top 100
# Or for a full dict with False for the rest:
tags = {a: (i < 100) for i, (a, _) in enumerate(sorted(scores.items(), key=lambda x: -x[1]))}
tagger.AssignTag("Mobile ISP", tags)

In [ ]:
for asn in top_100:
    print(asn, scores[asn])


In [ ]:
MOBILE_CORRECT = sorted({
    10396, 13771, 33392, 15344, 36511, 35900, 6639, 30689, 396357, 11594,
    40945, 32020, 11139, 11084, 17356, 33749, 25914, 394311, 3695, 30174,
    16696, 26288, 54614, 22324, 32307, 2740, 36827, 14813, 14593, 15146,
    396304, 46408, 14434, 3855, 46650, 40786, 22933, 21928, 8014, 14638,
    33576, 22581, 7922, 16705, 33582, 395561, 852, 21996, 36549, 22069,
    399724, 577, 11426, 11351, 33363, 46198, 10292, 7018, 701, 812, 7992,
    6327, 6167, 11290, 11427, 22773, 20001, 10796, 20115, 5769, 21744, 855,
    27653, 14754, 27775, 52253, 52233, 27745, 27725, 11816, 27651, 27781,
    8151, 27800, 27895, 8048, 52260, 27882, 6400, 11081, 27734, 27831,
    10620, 6147, 22047, 12252, 23201, 10269, 7418, 19863, 6057, 11556,
    7303, 27665, 27660, 28118, 52262, 18809, 27773, 6568, 27759, 28104,
    22085, 52228, 3816, 11830, 28036, 7122, 23243, 52362, 26611, 14709,
    52242, 262197, 27699, 13999, 11888, 28006, 28573, 25607, 26599, 4230,
    18881, 22927, 11172, 15221,
    # partial_correct: mobile (edge cases per notebook comment)
    8057, 22351,
    6453, # Tata, mobile-industry services (roaming/signalling/IPX)
    42334, 28229, 37098, 33125, 8818, 30990, 37461, 9751, 30986, 20776, 30985,
    37440, 7131, 15389, 8680, 327799, 38442, 12975, 37531, 29544, 36974,
    12709, 18001, 48252, 36890, 42961, 29465, 45245, 28885, 2860, 19037,
    49902, 37037, 133606, 9824, 24757, 8434, 55523, 139898, 36908, 15735,
    15706, 12430, 8346, 37342, 2516, 36935, 25543, 15805, 5466, 24432,
    204649, 55853, 29571, 6855, 45891, 38077, 16019, 45879, 25255, 17939,
    328191, 20634, 43940, 37577, 7522, 36873, 24439, 24389, 30992, 6758,
    45356, 4609, 23752, 29695, 37550, 31404, 28469, 30722, 4804, 17882,
    30987, 327934,
    28171 # partner
})

# Non-mobile (incorrect) - manually verified
# From: incorrect + small_neg_list + arin_non_mobile inline ["40627"], deduplicated
NON_MOBILE_INCORRECT = sorted({
    13335, 36290, 16413, 22995, 5650, 19323, 30165, 54665, 15305, 263763,
    27694, 27947, 52468, 27924, 52263, 21826, 17072, 272809, 27923, 265691,
    270963, 262146, 27729, 395555, 28219, 
    # small_neg_list (non-mobile; comment: "Tempest hosting, ...")
    209, 16437, 14061, 393275, 36231, 21859, 394684, 22363, 16509, 3549,
    19551, 8070, 19237, 268581, 
    # arin_non_mobile inline (when JSON not used) - explicitly non-mobile
    40627, 6939, 23520, 32098,
    19016, 265524, 16832, 30640, 134053, 15054, 39611, 46920, 38566, 262591, 31726, 198605,# new
    201004, 17501, 134371, 36924, 3356, 30058, 6677, 61461, 203214, 197558, 18049, 29256
})

In [ ]:
correct_mobile_cnt = 0
incorrect_mobile_cnt = 0
for asn in top_100:
    if int(asn) in MOBILE_CORRECT:
        correct_mobile_cnt += 1
    elif int(asn) in NON_MOBILE_INCORRECT:
        incorrect_mobile_cnt += 1
    else:
        print("need check", asn)

print(f"Correct mobile: {correct_mobile_cnt}")
print(f"Incorrect mobile: {incorrect_mobile_cnt}")


---
## Step 4: Performance Evaluation vs BGP.tools

To compare against BGP.tools, we collect all ASes that BGP.tools labels as **Mobile** (via the [mobile tag export](https://bgp.tools/tags/mobile.csv)). We manually check these ASes and add them to the same evaluation pool as our top-100 candidates, assigning each AS to either the **correct** (mobile) or **incorrect** (non-mobile) set defined above.

We then evaluate our inferred list and the BGP.tools list on this combined pool using precision, recall, and F1 score. Importantly, we **exclude the labeled ASes used to train our model** from the test set.

In [ ]:
import os
from pathlib import Path
import urllib.request

def _normalize_asn_str(asn) -> str:
    s = str(asn).strip().upper()
    return s[2:] if s.startswith("AS") else s

def load_bgp_tools_mobile(
    csv_path: str = "./data/bgp_tools/mobile.csv",
    fetch_if_missing: bool = True,
) -> list[int]:
    """Load BGP.tools mobile-tagged ASNs from local CSV or tag export."""
    path = Path(csv_path)
    if path.exists():
        lines = path.read_text().splitlines()
    elif fetch_if_missing:
        user_agent = os.environ.get(
            "BGP_TOOLS_USER_AGENT",
            "AS-Tagging semi_supervised_ml_tagging_example - zhiyichen@ipsec-10-2-0-182.vpn.gatech.edu",
        )
        req = urllib.request.Request(
            "https://bgp.tools/tags/mobile.csv",
            headers={"User-Agent": user_agent},
        )
        with urllib.request.urlopen(req) as resp:
            lines = resp.read().decode().splitlines()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines) + "\n")
    else:
        raise FileNotFoundError(f"Missing {csv_path} and fetch_if_missing=False")

    asns: list[int] = []
    for line in lines:
        if not line.strip():
            continue
        asn = _normalize_asn_str(line.split(",")[0])
        if asn.isdigit():
            asns.append(int(asn))
    # preserve order, drop duplicates
    return list(dict.fromkeys(asns))

bgp_tools_mobile_all = load_bgp_tools_mobile()
snapshot_asns = {int(k) for k in tagger.atomic_tags.keys()}
bgp_tools_mobile = [a for a in bgp_tools_mobile_all if a in snapshot_asns]

print(f"BGP.tools mobile (all tags): {len(bgp_tools_mobile_all)}")
print(f"BGP.tools mobile in snapshot: {len(bgp_tools_mobile)}")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

MOBILE_CORRECT_SET = set(MOBILE_CORRECT)
NON_MOBILE_INCORRECT_SET = set(NON_MOBILE_INCORRECT)

train_seeds = {int(a) for a in positive_asns} | {int(a) for a in negative_asns}
our_inferred = list(top_100.keys())

eval_pool = set(our_inferred) | set(bgp_tools_mobile)
eval_pool -= train_seeds

print(f"Training seeds excluded from evaluation: {len(train_seeds)}")
print(f"Evaluation pool size (top-100 ∪ BGP.tools mobile − train): {len(eval_pool)}")


def bin_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return {
        "precision": p,
        "recall": r,
        "f1": f1,
        "accuracy": acc,
        "n_labeled": len(y_true),
    }


def build_eval_vectors(inferred_list, pool, mobile_correct, non_mobile_incorrect):
    inferred_set = {str(int(a)) for a in inferred_list}
    y_true, y_pred, need_check = [], [], []
    for ai in pool:
        if ai in mobile_correct:
            yt = 1
        elif ai in non_mobile_incorrect:
            yt = 0
        else:
            need_check.append(ai)
            continue
        y_true.append(yt)
        y_pred.append(1 if str(ai) in inferred_set else 0)
    return y_true, y_pred, need_check


y_true_ours, y_pred_ours, need_check = build_eval_vectors(
    our_inferred,
    eval_pool,
    MOBILE_CORRECT_SET,
    NON_MOBILE_INCORRECT_SET,
)
res_ours = bin_metrics(y_true_ours, y_pred_ours)

y_true_bgp, y_pred_bgp, _ = build_eval_vectors(
    bgp_tools_mobile,
    eval_pool,
    MOBILE_CORRECT_SET,
    NON_MOBILE_INCORRECT_SET,
)
res_bgp = bin_metrics(y_true_bgp, y_pred_bgp)

print(f"ASes in pool with manual ground truth: {res_ours['n_labeled']}")
print(f"ASes in pool without labels (excluded from metrics): {len(need_check)}")
print()
print("Our inferred top-100:", res_ours)
print("BGP.tools mobile tag:", res_bgp)

In [ ]:
# Compare our top-100 against BGP.tools mobile tags (with manual ground truth)
our_set = {int(a) for a in our_inferred}
bgp_set = set(bgp_tools_mobile)

true_mobile = sorted(our_set & MOBILE_CORRECT_SET)
false_mobile = sorted(our_set & NON_MOBILE_INCORRECT_SET)
unlabeled = sorted(our_set - MOBILE_CORRECT_SET - NON_MOBILE_INCORRECT_SET)

print("Our top-100 — manual ground truth")
print(f"  True mobile (MOBILE_CORRECT):        {len(true_mobile)} / {len(our_set)}")
print(f"  Non-mobile (NON_MOBILE_INCORRECT):   {len(false_mobile)} / {len(our_set)}")
print(f"  Unlabeled (need manual check):       {len(unlabeled)} / {len(our_set)}")
if false_mobile:
    print(f"    non-mobile: {false_mobile}")
if unlabeled:
    print(f"    unlabeled: {unlabeled}")

print()
print("Our top-100 vs BGP.tools mobile tag")
in_bgp = sorted(our_set & bgp_set)
not_in_bgp = sorted(our_set - bgp_set)
print(f"  In BGP.tools mobile:     {len(in_bgp)} / {len(our_set)}")
print(f"  Not in BGP.tools mobile: {len(not_in_bgp)} / {len(our_set)}")

true_mobile_in_bgp = sorted(set(true_mobile) & bgp_set)
true_mobile_not_in_bgp = sorted(set(true_mobile) - bgp_set)
false_mobile_in_bgp = sorted(set(false_mobile) & bgp_set)
false_mobile_not_in_bgp = sorted(set(false_mobile) - bgp_set)

print()
print("  By ground truth:")
print(f"    true mobile in BGP.tools:     {len(true_mobile_in_bgp)} / {len(true_mobile)}")
print(f"    true mobile not in BGP.tools: {len(true_mobile_not_in_bgp)} / {len(true_mobile)}")
print(f"    non-mobile in BGP.tools:      {len(false_mobile_in_bgp)} / {len(false_mobile)}")
print(f"    non-mobile not in BGP.tools:  {len(false_mobile_not_in_bgp)} / {len(false_mobile)}")
if false_mobile_in_bgp:
    print(f"      non-mobile but tagged mobile by BGP.tools: {false_mobile_in_bgp}")
if false_mobile_not_in_bgp:
    print(f"      non-mobile and not in BGP.tools: {false_mobile_not_in_bgp}")

print()
print("BGP.tools mobile tag quality (manual labels)")
bgp_incorrect = sorted(bgp_set & NON_MOBILE_INCORRECT_SET)
bgp_not_incorrect = sorted(bgp_set - NON_MOBILE_INCORRECT_SET)
print(f"  Incorrect (in NON_MOBILE_INCORRECT): {len(bgp_incorrect)} / {len(bgp_set)}")
print(f"  Not marked incorrect:                {len(bgp_not_incorrect)} / {len(bgp_set)}")
if bgp_incorrect:
    print(f"    incorrect: {bgp_incorrect}")

In [ ]:
# Diagnostics: overlap between BGP.tools mobile list and training seeds
bgp_in_train = sorted(set(bgp_tools_mobile) & train_seeds)
print(f"BGP.tools mobile ASes that are training seeds: {len(bgp_in_train)}")
if bgp_in_train:
    print(bgp_in_train)

# Optional: compare inferred lists
our_only = set(our_inferred) - set(bgp_tools_mobile)
bgp_only = set(bgp_tools_mobile) - set(our_inferred) - train_seeds
print(f"Our top-100 not in BGP.tools mobile: {len(our_only)}")
print(f"BGP.tools mobile not in our top-100 (excl. train): {len(bgp_only)}")

# Sanity: pool members missing manual labels (should be reviewed over time)
if need_check:
    print(f"\nSample pool ASes without manual label ({len(need_check)} total):")
    print(sorted(int(a) for a in need_check)[:20])

In [ ]:
for asn in top_100:
    print(asn, scores[asn])
